
# 🔄 Prefect Notes 2.0 — Production Architecture
> **Focus:** How Prefect runs in real enterprise production environments
> **Stack:** Prefect + Kubernetes + Docker + GitLab + Rancher + OpenSearch + Grafana + JFrog

---

## 1. 🏭 Production Stack Overview

In production, Prefect doesn't run alone — it's part of a full infrastructure stack. Here's every tool and what it does:

| Tool | Role | Think of it as |
|---|---|---|
| **Prefect** | Workflow orchestration | The brain — decides what runs, when, and tracks it |
| **Kubernetes (K8s)** | Execution environment | The factory floor — actually runs the work |
| **Docker** | Containerization | The box that packages your code + dependencies |
| **GitLab** | Source code storage | Where your flow code lives |
| **Rancher** | Kubernetes GUI | Visual dashboard to see what's running on K8s |
| **OpenSearch** | Log storage & search | Where all logs go — search and debug issues |
| **Grafana** | Metrics monitoring | CPU, memory, health dashboards |
| **JFrog Artifactory** | Artifact storage | Stores Docker images and build packages |

---

## 2. 🧠 Prefect Core Concepts Refresher (Production Context)

### Flows
- A Flow = the entire workflow
- Collection of tasks that run in a defined order or dependency structure
- In production, flows run inside **Kubernetes pods** — not on your laptop

### Deployments
A Deployment is a **packaged configuration** that tells Prefect how to run a flow:

```
Deployment defines:
  ✅ Which flow to run
  ✅ Where the code is stored (GitLab via Storage Block)
  ✅ Which infrastructure runs it (Kubernetes via Work Pool)
  ✅ When to trigger it (schedule / event / manual)
  ✅ What parameters to pass
```

> Think of a Deployment as a **recipe card** — the flow is the dish, the deployment is the instructions for making it in a specific kitchen (Kubernetes).

### Work Pools
- Define **where** flows execute
- In production → Work Pool is configured for **Kubernetes**
- Prefect tells Kubernetes: "create a job, run this flow"

---

## 3. ⚙️ Full Execution Process in Production

When a flow is triggered, here's exactly what happens step by step:

```
TRIGGER (schedule / manual / event)
         ↓
Prefect Server receives flow run request
         ↓
Reads Deployment configuration
         ↓
Work Pool → selects Kubernetes as infrastructure
         ↓
Kubernetes Job is created
         ↓
Job launches a Pod
         ↓
Pod fetches flow code from GitLab (via Storage Block)
         ↓
Pod runs the Prefect flow inside Docker container
         ↓
Flow completes → Pod terminates
         ↓
Logs → OpenSearch
Metrics → Grafana
Results → Prefect UI
```

---

## 4. ☸️ Kubernetes — What You Need to Know

Kubernetes (K8s) is the execution engine for Prefect flows in production. Here are the key components:

### Pods
- **Smallest execution unit** in Kubernetes
- Each Prefect flow run gets **its own pod**
- Pod starts → flow runs → pod terminates
- Isolated — one flow's crash doesn't affect another

### Jobs
- Kubernetes **Jobs** are used to run tasks once and complete
- Prefect creates a Kubernetes Job for each flow run
- Job manages the pod lifecycle (create → run → cleanup)

### Services
- Enable **communication between components** inside the cluster
- Example: Prefect server communicates with workers via a Service

### Ingress
- Exposes internal services to the **outside world** via a domain name
- Example from production: `chat.ai.avl.com`
- Routes external HTTP traffic → correct internal service

### Architecture Diagram:

```
External Request
      ↓
   Ingress (domain: chat.ai.avl.com)
      ↓
   Service (internal routing)
      ↓
   ┌─────────────────────────────────┐
   │         Kubernetes Cluster      │
   │                                 │
   │  ┌──────────┐  ┌──────────┐    │
   │  │  Pod 1   │  │  Pod 2   │    │
   │  │ Flow Run │  │ Flow Run │    │
   │  │ (Docker) │  │ (Docker) │    │
   │  └──────────┘  └──────────┘    │
   │                                 │
   │  Prefect Server (always on)    │
   └─────────────────────────────────┘
```

---

## 5. 🐳 Docker — Containerization

Every Prefect flow in production runs inside a **Docker container**. The Docker image contains:

```
Docker Image = 
  Python runtime
  + All pip dependencies (requirements.txt)
  + Flow code
  + Environment config
```

### Why Docker?
- **Consistency** — same environment everywhere (dev, staging, prod)
- **Isolation** — one flow's dependencies don't conflict with another's
- **Reproducibility** — exact same image = exact same behavior every time

### Flow in Production:
```
Developer writes flow code
        ↓
Docker image built (code + deps packaged)
        ↓
Image pushed to JFrog Artifactory
        ↓
Kubernetes pulls image from JFrog
        ↓
Container runs the flow
```

---

## 6. 📦 Storage Blocks & Infrastructure Blocks

### Storage Blocks
Define **where the flow code is stored** and how Prefect retrieves it.

In this production setup:
- Code lives in **GitLab** repositories
- Storage Block = GitLab connector
- When a flow runs, Prefect fetches latest code from GitLab automatically

Storage blocks can also store sensitive info:
```
✅ API keys
✅ Database passwords
✅ Cloud credentials
✅ Secrets
```

### Infrastructure Blocks
Define **how and where flows execute**.

| Block Type | Runs on |
|---|---|
| Kubernetes | Kubernetes cluster |
| Docker | Docker container locally |
| Process | Local machine process |

In production → **Kubernetes Infrastructure Block** configures:
- Pod resource limits (CPU, memory)
- Which Docker image to use
- Environment variables
- Namespace

---

## 7. 🦊 GitLab Integration

GitLab is the **single source of truth** for all flow code in production.

```
Developer pushes code to GitLab
        ↓
Prefect Deployment references GitLab repo
        ↓
Flow triggered (schedule / manual)
        ↓
Storage Block fetches latest code from GitLab
        ↓
Kubernetes pod executes that code
```

**Why this matters:**
- No code baked into the server — always fetched fresh from GitLab
- Update the code in GitLab → next run automatically uses new version
- Full version history of all pipeline code

---

## 8. 🖥️ Rancher — Kubernetes GUI

Rancher is a **visual management interface** for Kubernetes — like a dashboard for the cluster.

What you can do in Rancher:

```
✅ View all running pods
✅ See Kubernetes Jobs (each = one flow run)
✅ Inspect pod logs in real-time
✅ Monitor cluster resource usage (CPU, memory per pod)
✅ Restart or delete pods
✅ See why a pod failed (OOMKilled, CrashLoopBackOff etc.)
```

**In production workflow:**
When a Prefect flow run starts → go to Rancher → find the pod → inspect logs if something is wrong.

---

## 9. 📊 Monitoring & Logging Stack

### Grafana — System Metrics
Grafana shows **infrastructure-level metrics** in visual dashboards:

```
What Grafana monitors:
  📈 CPU usage per pod / node
  📈 Memory consumption
  📈 Network throughput
  📈 System health over time
  📈 Alerts when thresholds breached
```

> Think of Grafana as the **health monitor** — is the system running well?

### OpenSearch — Logs
OpenSearch stores and makes **all logs searchable**:

```
What OpenSearch stores:
  📋 Pod logs (stdout/stderr from your flow)
  📋 Application-level logs
  📋 Error stack traces
  📋 Historical log search
```

> Think of OpenSearch as the **black box recorder** — what actually happened inside the flow?

### Grafana vs OpenSearch:

| | Grafana | OpenSearch |
|---|---|---|
| **Type** | Metrics (numbers, charts) | Logs (text, events) |
| **Use for** | Is the system healthy? | What happened in this run? |
| **Example** | Pod using 90% memory | "KeyError on line 42 of transform.py" |

---

## 10. 🗄️ JFrog Artifactory — Artifact Storage

JFrog is the **central repository** for all build artifacts:

```
Stores:
  🐳 Docker images (your flow containers)
  📦 Python packages
  🔧 Build dependencies
  📁 Release artifacts
```

**In the pipeline:**
```
Build Docker image locally
        ↓
Push to JFrog Artifactory
        ↓
Kubernetes pulls from JFrog when creating pods
        ↓
Flow runs inside that container
```

---

## 11. 💀 Kubernetes Failures — What to Know

### OOMKilled — Most Common Issue

**OOM = Out Of Memory**

```
What happens:
Pod starts → Flow runs → Data gets large → Pod exceeds memory limit
→ Kubernetes terminates the pod → Status: OOMKilled
```

**How to diagnose in Rancher:**
```
Rancher → Find the pod → Status = OOMKilled
        ↓
Check memory limit on the pod (e.g. 512Mi)
        ↓
Either: increase memory limit in Infrastructure Block
    OR: optimize the flow (process in chunks, don't load full dataset)
```

### Other Common Failures:

| Error | Cause | Fix |
|---|---|---|
| `OOMKilled` | Pod exceeded memory limit | Increase memory limit or optimize code |
| `CrashLoopBackOff` | Pod keeps crashing on start | Check logs — usually a code/dependency error |
| `ImagePullBackOff` | Can't pull Docker image from JFrog | Check image name/tag, JFrog credentials |
| `Pending` (forever) | Not enough cluster resources | Scale cluster or reduce resource requests |

---

## 12. ⌨️ kubectl — Kubernetes CLI

`kubectl` is the command-line tool to manage Kubernetes directly — useful when Rancher UI isn't enough.

```bash
# See all running pods
kubectl get pods

# See pods in a specific namespace
kubectl get pods -n prefect

# See logs of a specific pod (your flow's output)
kubectl logs pod-name-abc123

# Follow logs in real-time
kubectl logs -f pod-name-abc123

# Describe a pod (see why it failed)
kubectl describe pod pod-name-abc123

# See all jobs
kubectl get jobs

# Delete a stuck pod
kubectl delete pod pod-name-abc123

# See resource usage per pod
kubectl top pods
```

---

## 13. 🌐 Full System Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                        PRODUCTION SYSTEM                     │
│                                                              │
│  Developer                                                   │
│      ↓  pushes code                                          │
│  GitLab (code repository)                                    │
│      ↓  storage block fetches                                │
│  Prefect Server (on Kubernetes — always running)             │
│      ↓  creates flow run                                     │
│  Work Pool (Kubernetes)                                      │
│      ↓  creates job                                          │
│  Kubernetes Job                                              │
│      ↓  launches pod                                         │
│  Pod (Docker container from JFrog)                           │
│      ↓  executes flow                                        │
│  Flow runs + completes                                       │
│      ↓  outputs go to                                        │
│  ┌──────────────┐  ┌─────────────┐  ┌──────────────┐       │
│  │  OpenSearch  │  │   Grafana   │  │  Prefect UI  │       │
│  │  (logs)      │  │  (metrics)  │  │  (flow runs) │       │
│  └──────────────┘  └─────────────┘  └──────────────┘       │
│                                                              │
│  Rancher → visual management of all the above               │
└──────────────────────────────────────────────────────────────┘
```

---

## 14. 📌 Quick Recap

```
Prefect          → orchestrates workflows (flows, tasks, deployments)
Kubernetes       → runs flows inside pods, one pod per flow run
Docker           → packages code + deps into portable container
GitLab           → stores flow code, fetched via Storage Block
Storage Block    → tells Prefect where to get code (GitLab)
Infra Block      → tells Prefect where to run code (Kubernetes)
Work Pool        → queue that connects deployments to workers
Rancher          → GUI to see pods, jobs, logs on Kubernetes
OpenSearch       → search and analyze logs from flow runs
Grafana          → monitor CPU/memory/health of infrastructure
JFrog            → stores Docker images and build artifacts
kubectl          → CLI to manage Kubernetes directly

OOMKilled        → pod ran out of memory → increase limit or optimize code
```

---

## 15. 📚 Recommended Learning Resources

To understand Kubernetes and Docker deeper:

- **TechWorld with Nana** (YouTube) — best channel for Docker + Kubernetes from scratch
  - Docker basics → containerization concepts
  - Kubernetes architecture → pods, services, ingress explained visually
- **Prefect Docs** → docs.prefect.io
- **Kubernetes Docs** → kubernetes.io/docs

---

*Prefect Notes 2.0 ✅ — Production Architecture*
*Next: Hands-on with deployments, work pools and kubectl commands*